# SOC Validator — Interactive Jupyter Widget

This notebook builds a self-contained `ipywidgets` UI for the **structural-isomorphism** `soc_pipeline`.

**What it does**

1. Pick a dataset (bundled, file upload, or paste raw numbers).
2. Tweak parameters (xmin override, bootstrap n, expected band).
3. Click **Run validation** → live verdict + CCDF plot + LR test numbers.

**Audience.** Working scientists who want to drop their own time-series into a self-organized criticality (SOC) check without learning the API.

**Why this matters.** The pipeline takes 5 lines of code, but a UI shortens the loop from minutes to seconds when you want to swap datasets, sweep `n_boot`, or argue with a band.


## 1. Imports

In [ ]:
from __future__ import annotations
import io
import json
import sys
from pathlib import Path

# If running from inside a worktree, prefer the local src/ over any
# globally-installed (possibly stale) soc-pipeline editable install.
_here = Path('.').resolve()
for _ in range(6):
    _candidate = _here / 'packages' / 'soc-pipeline' / 'src'
    if _candidate.is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
    _here = _here.parent

# Drop any pre-imported soc_pipeline so the new path wins
for _mod in list(sys.modules):
    if _mod == 'soc_pipeline' or _mod.startswith('soc_pipeline.'):
        del sys.modules[_mod]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

import soc_pipeline
from soc_pipeline import validate, empirical_ccdf

print('soc_pipeline version:', soc_pipeline.__version__)
print('soc_pipeline path:   ', soc_pipeline.__file__)


## 2. Dataset loader

We bundle 3 lightweight datasets in `dataset/v1/`. The loader returns a flat `numpy.array` of positive event sizes.

In [ ]:
REPO_ROOT = Path('.').resolve()
while REPO_ROOT.name and not (REPO_ROOT / 'dataset' / 'v1').is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATASETS = {
    'earthquake-magnitudes':  REPO_ROOT / 'dataset/v1/systems/01_earthquake/data/catalog.jsonl',
    'wildfire-sizes':         REPO_ROOT / 'dataset/v1/systems/05_wildfire/data/fires.jsonl',
    'synthetic-pareto-alpha-2.5': None,  # generated on demand
    'synthetic-lognormal':        None,
}

EXPECTED_BANDS = {
    'earthquake-magnitudes':       (2.5, 3.5),   # Gutenberg-Richter regime
    'wildfire-sizes':              (1.8, 2.5),
    'synthetic-pareto-alpha-2.5':  (2.3, 2.7),
    'synthetic-lognormal':         (2.0, 3.0),
}

def load_dataset(name: str) -> np.ndarray:
    if name == 'synthetic-pareto-alpha-2.5':
        rng = np.random.default_rng(0)
        return rng.pareto(1.5, 5000) + 1.0
    if name == 'synthetic-lognormal':
        rng = np.random.default_rng(0)
        return np.exp(rng.normal(0, 1, 5000))
    path = DATASETS[name]
    if path is None or not path.exists():
        raise FileNotFoundError(f'Dataset {name} not found at {path}')
    records = []
    with path.open() as f:
        for line in f:
            if not line.strip():
                continue
            r = json.loads(line)
            if name == 'earthquake-magnitudes':
                if r.get('mag') is not None:
                    records.append(float(r['mag']))
            elif name == 'wildfire-sizes':
                if r.get('size_acres') is not None:
                    records.append(float(r['size_acres']))
    arr = np.asarray(records, dtype=float)
    arr = arr[np.isfinite(arr) & (arr > 0)]
    return arr

def parse_text(text: str) -> np.ndarray:
    nums = []
    for tok in text.replace(',', ' ').replace('\n', ' ').split():
        try:
            nums.append(float(tok))
        except ValueError:
            continue
    return np.asarray(nums, dtype=float)

def parse_uploaded(content: bytes) -> np.ndarray:
    text = content.decode('utf-8', errors='ignore')
    # Try JSON-lines first
    try:
        recs = [json.loads(l) for l in text.splitlines() if l.strip()]
        if recs and isinstance(recs[0], dict):
            # auto-pick first numeric field
            for k, v in recs[0].items():
                if isinstance(v, (int, float)):
                    return np.asarray([float(r[k]) for r in recs if r.get(k) is not None])
    except (json.JSONDecodeError, KeyError):
        pass
    return parse_text(text)

# Quick smoke-test the loader
_demo = load_dataset('synthetic-pareto-alpha-2.5')
print(f'synthetic-pareto-alpha-2.5: n={len(_demo)}, head={_demo[:3]}')

## 3. Input controls

Three input modes: bundled dataset, file upload, paste text. We pick the active mode with a `Tab` widget.

In [ ]:
dataset_dropdown = widgets.Dropdown(
    options=list(DATASETS.keys()),
    value='synthetic-pareto-alpha-2.5',
    description='Dataset:',
    layout=widgets.Layout(width='400px'),
)

file_uploader = widgets.FileUpload(
    accept='.jsonl,.json,.csv,.txt',
    multiple=False,
    description='Upload file',
)

paste_box = widgets.Textarea(
    placeholder='Paste numbers separated by whitespace, commas, or newlines',
    layout=widgets.Layout(width='600px', height='150px'),
)

input_tabs = widgets.Tab(children=[dataset_dropdown, file_uploader, paste_box])
input_tabs.set_title(0, 'Bundled')
input_tabs.set_title(1, 'Upload')
input_tabs.set_title(2, 'Paste')

## 4. Parameter controls

Bootstrap N, expected α band (two FloatText widgets), and an optional xmin override.

The xmin slider has a checkbox guard — by default we let Clauset KS-minimization pick xmin. Tick the box to override.

In [ ]:
n_boot_slider = widgets.IntSlider(
    value=100, min=0, max=500, step=50,
    description='n_boot:',
    continuous_update=False,
)

band_low = widgets.FloatText(value=2.3, description='band low:', layout=widgets.Layout(width='180px'))
band_high = widgets.FloatText(value=2.7, description='band high:', layout=widgets.Layout(width='180px'))
band_box = widgets.HBox([band_low, band_high])

xmin_use = widgets.Checkbox(value=False, description='override xmin')
xmin_slider = widgets.FloatSlider(
    value=1.0, min=0.01, max=100.0, step=0.1,
    description='xmin:',
    readout_format='.2f',
    continuous_update=False,
)
xmin_box = widgets.HBox([xmin_use, xmin_slider])

run_button = widgets.Button(
    description='Run validation',
    button_style='primary',
    icon='play',
)

params_panel = widgets.VBox([n_boot_slider, band_box, xmin_box, run_button])

## 5. Output area

We build a verdict label with a traffic-light color, a textual summary, and an Output widget for the matplotlib CCDF plot.

In [ ]:
verdict_label = widgets.HTML(
    value="<h2 style='color:#666'>Waiting for run…</h2>",
)

summary_html = widgets.HTML(value='')
plot_output = widgets.Output()

output_panel = widgets.VBox([verdict_label, summary_html, plot_output])

VERDICT_COLOR = {
    'PASS': '#2ecc71',
    'FAIL': '#e74c3c',
    'INCONCLUSIVE': '#f39c12',
    'ERROR': '#7f8c8d',
}

def render_verdict(v):
    if v.error is not None:
        color = VERDICT_COLOR['ERROR']
        verdict_label.value = f"<h2 style='color:{color}'>ERROR — {v.error}</h2>"
    else:
        color = VERDICT_COLOR.get(v.verdict, VERDICT_COLOR['INCONCLUSIVE'])
        verdict_label.value = f"<h2 style='color:{color}'>{v.verdict} — {v.label}</h2>"
    ci = (
        f"CI=[{v.alpha_ci_low:.3f}, {v.alpha_ci_high:.3f}]"
        if v.alpha_ci_low is not None else 'CI=skipped'
    )
    summary_html.value = (
        f"<pre style='background:#fafafa;padding:8px;font-size:13px'>"
        f"alpha       = {v.alpha if v.alpha is None else f'{v.alpha:.3f}'}\n"
        f"            {ci}\n"
        f"xmin        = {v.xmin if v.xmin is None else f'{v.xmin:.3f}'}\n"
        f"n_tail/n    = {v.n_tail}/{v.n_total}\n"
        f"KS distance = {v.ks_statistic if v.ks_statistic is None else f'{v.ks_statistic:.4f}'}\n"
        f"vs LN  R={v.vs_lognormal_R}  p={v.vs_lognormal_p}\n"
        f"vs EXP R={v.vs_exponential_R}  p={v.vs_exponential_p}\n"
        f"in_band     = {v.in_band}\n"
        f"underlying  = {v.underlying_verdict}"
        f"</pre>"
    )

## 6. CCDF plot helper

In [ ]:
def plot_ccdf(arr: np.ndarray, alpha: float | None, xmin: float | None) -> None:
    plot_output.clear_output(wait=True)
    with plot_output:
        grid, ccdf = empirical_ccdf(arr, n_points=100)
        if grid is None:
            print('CCDF: no positive data')
            return
        fig, ax = plt.subplots(figsize=(6.5, 4))
        mask = ccdf > 0
        ax.loglog(grid[mask], ccdf[mask], 'o', markersize=3, alpha=0.6, label='empirical')
        if alpha is not None and xmin is not None and xmin > 0:
            # Power-law tail overlay: P(X>x) = (x/xmin)^(1-alpha) for x>=xmin
            tail = grid[grid >= xmin]
            if len(tail) >= 2:
                idx = np.argmin(np.abs(grid - xmin))
                base = ccdf[idx] if ccdf[idx] > 0 else 1e-3
                pl = base * (tail / xmin) ** (1 - alpha)
                ax.loglog(tail, pl, '-', lw=2, color='#e74c3c',
                          label=f'fit α={alpha:.2f}')
                ax.axvline(xmin, color='gray', ls='--', alpha=0.5,
                           label=f'xmin={xmin:.2f}')
        ax.set_xlabel('event size')
        ax.set_ylabel('P(X > x)')
        ax.set_title('Complementary CDF')
        ax.legend(fontsize=9)
        ax.grid(True, which='both', alpha=0.3)
        plt.tight_layout()
        plt.show()

## 7. Wire the Run button

On click: pull the active input mode, run `validate()`, render.

In [ ]:
def get_active_data() -> tuple[np.ndarray, str]:
    idx = input_tabs.selected_index
    if idx == 0:
        name = dataset_dropdown.value
        return load_dataset(name), name
    if idx == 1:
        uploads = file_uploader.value
        if not uploads:
            raise ValueError('No file uploaded')
        # ipywidgets 8.x: value is a tuple of dicts
        first = uploads[0] if isinstance(uploads, (list, tuple)) else list(uploads.values())[0]
        content = first['content'] if 'content' in first else first
        name = first.get('name', 'uploaded')
        return parse_uploaded(bytes(content)), name
    text = paste_box.value.strip()
    if not text:
        raise ValueError('Paste box is empty')
    return parse_text(text), 'pasted'

def on_run_click(b):  # noqa: ARG001 — ipywidgets handler signature
    try:
        arr, name = get_active_data()
    except Exception as exc:
        verdict_label.value = f"<h2 style='color:#e74c3c'>Input error — {exc}</h2>"
        summary_html.value = ''
        plot_output.clear_output()
        return
    if len(arr) == 0:
        verdict_label.value = "<h2 style='color:#e74c3c'>No positive numeric data parsed</h2>"
        return
    expected = (float(band_low.value), float(band_high.value))
    v = validate(
        arr, label=name, expected_band=expected,
        n_boot=int(n_boot_slider.value),
    )
    render_verdict(v)
    plot_ccdf(arr, v.alpha, v.xmin)

run_button.on_click(on_run_click)

## 8. Final assembled widget

Display the full UI. Interact in-place.

In [ ]:
header = widgets.HTML(
    "<h1>SOC Validator</h1>"
    "<p style='color:#666'>Drop in a time-series of event sizes → get a power-law verdict in seconds.</p>"
)

app = widgets.VBox([
    header,
    widgets.HTML('<h3>1. Input</h3>'),
    input_tabs,
    widgets.HTML('<h3>2. Parameters</h3>'),
    params_panel,
    widgets.HTML('<h3>3. Result</h3>'),
    output_panel,
])

display(app)

# Auto-run on a quick synthetic so the notebook executes cleanly via nbconvert.
on_run_click(None)